# Logging and Run Summaries

This notebook adds logging and summaries to all data processing steps.

Logging includes:
- source table name
- target table name
- rows read
- rows written
- aggregation grain
- business metrics created

Goal:
Improve observability, monitoring, and debugging.

In [0]:
from pyspark.sql.functions import *

catalog = "vattenfall_dev"
refined_schema = "refined"
gold_schema = "gold"

In [0]:
# Cell 1: Imports and Config
import yaml
from pyspark.sql import functions as F

# Path to your project config
config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"

# Load YAML config
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Extract catalog and schema info
catalog = config["catalog"]
raw_schema = config["schemas"]["raw"]
silver_schema = "refined"

In [0]:
# Cell 2: Silver Tables List
silver_tables = [
    "silver_market_prices",
    "silver_weather",
    "silver_grid_events",
    "silver_asset_reference",  # optional
    "silver_regional_operations_base"  # optional
]

In [0]:
# Cell 3: Map Silver tables to Gold tables and business metrics
gold_mapping = {
    "silver_market_prices": {
        "target": "gold_daily_market_summary",
        "grain": ["market_type", "report_day"],
        "metric": "avg_price"
    },
    "silver_weather": {
        "target": "gold_weather_impact_summary",
        "grain": ["region", "report_day"],
        "metric": "weather_impact_score"
    },
    "silver_grid_events": {
        "target": "gold_grid_incident_summary",
        "grain": ["region", "event_date"],
        "metric": "incident_severity_score"
    },
    "silver_regional_operations_base": {
        "target": "gold_regional_operations_dashboard",
        "grain": ["region", "report_day"],
        "metric": "regional_operational_index"
    },
    # silver_asset_reference may not have a direct Gold table
}

In [0]:
df_silver.printSchema()
df_silver.show(5, truncate=False)

In [0]:
# ==========================
# Milestone 2: Logging and Run Summaries
# ==========================

# Table names
source_table = f"{catalog}.{silver_schema}.silver_grid_events"
target_table = f"{catalog}.gold.gold_grid_incident_summary"

# Read Silver table
df_silver = spark.table(source_table)

# Define aggregation grain and placeholder business metric
grain = ["event_day", "region"]
metric = "incident_count"  # placeholder metric

# Count rows read
rows_read = df_silver.count()

# Build placeholder Gold dataframe
df_gold = df_silver.groupBy(*grain).agg(
    F.lit(0).alias(metric)  # replace 0 with actual metric calculation later
)

# Count rows written
rows_written = df_gold.count()

# Display logging info
print(f"Source Table: {source_table}")
print(f"Target Table: {target_table}")
print(f"Rows Read: {rows_read}")
print(f"Rows Written: {rows_written}")
print(f"Aggregation Grain: {', '.join(grain)}")
print(f"Business Metric Created: {metric}")